<a href="https://colab.research.google.com/github/BlairRong/Python-AI-course-lab-Cloud/blob/main/4.9_lab_CosmosDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install azure-cosmos

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.9/424.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 8.2 MB/s eta 0:00:00


Design Data Model

In [ ]:
{
  "id": "order_001",
  "userId": "user123",
  "userName": "Siying Rong",
  "userEmail": "siying@example.com",
  "items": [
    { "productId": "p1", "name": "Laptop", "price": 1200, "quantity": 1 },
    { "productId": "p2", "name": "Mouse", "price": 25, "quantity": 2 }
  ],
  "total": 1250,
  "status": "shipped",
  "createdAt": "2026-04-09T10:00:00Z"
}

{'id': 'order_001',
 'userId': 'user123',
 'userName': 'Siying Rong',
 'userEmail': 'siying@example.com',
 'items': [{'productId': 'p1', 'name': 'Laptop', 'price': 1200, 'quantity': 1},
  {'productId': 'p2', 'name': 'Mouse', 'price': 25, 'quantity': 2}],
 'total': 1250,
 'status': 'shipped',
 'createdAt': '2026-04-09T10:00:00Z'}

Part3 - Data Modeling

Insert 10-20 items
Use realistic data
Include: Nested objects, Arrays

In [ ]:
from azure.cosmos import CosmosClient
import datetime, random
import os
os.environ['COSMOS_DB_URL'] = " "
os.environ['COSMOS_DB_KEY'] = " "

url = os.environ['COSMOS_DB_URL']
key = os.environ['COSMOS_DB_KEY']

client = CosmosClient(url, credential=key)
database = client.get_database_client("ecommerceDB")
container = database.get_container_client("orders")

users = ["user001", "user002", "user003"]
statuses = ["pending", "shipped", "delivered", "cancelled"]
products = [
    {"id": "p1", "name": "Laptop", "price": 1200},
    {"id": "p2", "name": "Mouse", "price": 25},
    {"id": "p3", "name": "Keyboard", "price": 80},
    {"id": "p4", "name": "Monitor", "price": 300},
]

def generate_order(i):
    user = random.choice(users)
    num_items = random.randint(1, 3)
    items = []
    total = 0
    for _ in range(num_items):
        prod = random.choice(products)
        qty = random.randint(1, 2)
        items.append({"productId": prod["id"], "name": prod["name"], "price": prod["price"], "quantity": qty})
        total += prod["price"] * qty
    return {
        "id": f"order_{i}",
        "userId": user,
        "userName": f"User {user[-3:]}",
        "userEmail": f"{user}@example.com",
        "items": items,
        "total": total,
        "status": random.choice(statuses),
        "createdAt": datetime.datetime.utcnow().isoformat() + "Z"
    }

for i in range(1, 16):
    container.upsert_item(generate_order(i))
    print(f"Inserted order {i}")

/tmp/ipykernel_2661/3991773435.py:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "createdAt": datetime.datetime.utcnow().isoformat() + "Z"


Inserted order 1
Inserted order 2
Inserted order 3
Inserted order 4
Inserted order 5
Inserted order 6
Inserted order 7
Inserted order 8
Inserted order 9
Inserted order 10
Inserted order 11
Inserted order 12
Inserted order 13
Inserted order 14
Inserted order 15


Part 4 - Queries 查询
create and run 5 queries

In [ ]:
# 1. Basic SELECT 全查询
all_orders = list(container.query_items(query="SELECT * FROM c", enable_cross_partition_query=True))
print("Total orders:", len(all_orders))

# 2. Filtering 过滤 WHERE
user_orders = list(container.query_items(query="SELECT * FROM c WHERE c.userId = 'user001'", enable_cross_partition_query=True))
print("User001 orders:", len(user_orders))

# 3. Sorting 排序 ORDER BY
sorted_orders = list(container.query_items(query="SELECT * FROM c ORDER BY c.total DESC", enable_cross_partition_query=True))
for o in sorted_orders[:3]:
    print(o["id"], o["total"])

# 4. Aggregation 聚合 COUNT or similar
from collections import Counter

# 假设 all_orders 已经包含了所有订单（前面已经执行过）
user_counts = Counter([item['userId'] for item in all_orders])
print("Count per user:", [{"userId": uid, "orderCount": cnt} for uid, cnt in user_counts.items()])

# 5. Query on nested data 查询嵌套数组（包含 Laptop）
laptop_orders = list(container.query_items(query="SELECT * FROM c WHERE ARRAY_CONTAINS(c.items, {'name': 'Laptop'}, true)", enable_cross_partition_query=True))
print("Orders with Laptop:", len(laptop_orders))

Total orders: 15
User001 orders: 3
order_8 6000
order_3 3080
order_4 2640
Count per user: [{'userId': 'user003', 'orderCount': 8}, {'userId': 'user002', 'orderCount': 4}, {'userId': 'user001', 'orderCount': 3}]
Orders with Laptop: 7


Part 5 - Python Integration

In [ ]:
# write a new order 写一条新订单
new_order = {
    "id": "order_16",
    "userId": "user004",
    "userName": "New User",
    "userEmail": "new@example.com",
    "items": [{"productId": "p5", "name": "Tablet", "price": 500, "quantity": 1}],
    "total": 500,
    "status": "pending",
    "createdAt": datetime.datetime.utcnow().isoformat() + "Z"
}
container.upsert_item(new_order)
# Read this order 读该订单
read_item = container.read_item(item="order_16", partition_key="user004")
print("Read item:", read_item["id"])

/tmp/ipykernel_2661/735526792.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "createdAt": datetime.datetime.utcnow().isoformat() + "Z"


Read item: order_16


Part 6 - Reflection

1.Why structure data this way?

Embedded items array avoids joins; duplicated user info speeds up queries.

2.Why choose partition key /userId?

Most queries are “orders by user”; keeps all orders of one user together for efficient retrieval.

3.Challenges?

Learning ARRAY_CONTAINS syntax; managing keys securely.

4.Scaling to millions of users?

Partitions scale automatically; risk of hot partition for very active users – could use composite key.

5.Advantages over relational DB?

Flexible schema, global distribution, automatic scaling, no need for migrations.



为什么要这样组织数据？

嵌入式项目数组避免了连接操作；重复的用户信息可以加快查询速度。

为什么选择分区键 /userId？

大多数查询都是“按用户排序”；将同一用户的所有订单集中在一起，可以提高检索效率。

挑战？

学习 ARRAY_CONTAINS 语法；安全地管理键。

如何扩展到数百万用户？

分区可以自动扩展；但对于非常活跃的用户，存在热分区的风险——可以使用复合键。

相比关系型数据库的优势？

灵活的模式、全球分布、自动扩展、无需迁移。